BISMILLAH IRRAKMA NIR RAHEEM

In [8]:
import joblib
import pandas as pd
import random

#  Load the real model
model = joblib.load('t2_strategy_model.pkl')

# Let's test it immediately to make sure it works
print("Model Type:", type(model))

print("✅ Model loaded and ready for simulation!")


Model Type: <class 'sklearn.pipeline.Pipeline'>
✅ Model loaded and ready for simulation!


In [9]:
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['TyreLife','TyreLifePercent','Avg_TrackTemp',...,'Compound', 'CircuitArchetype','StintPhase']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This su

In [10]:
# TESTING THE INCOOPERATION AS WELL AS THE MODEL AND NON MODEL CAR.

In [11]:
def run_phase5_simulation(total_laps, pit_lap_A, pit_lap_B, num_simulations=100):
    wins_A = 0
    wins_B = 0
    
    for sim in range(num_simulations):
        tyre_age_A = 0
        total_time_A = 0.0
        
        tyre_age_B = 0
        total_time_B = 0.0
        
        for lap in range(1, total_laps + 1):
            
            # --- CAR A (YOU) USES THE ML MODEL ---
            # 1. Build the exact same features the model was trained on
            race_state_A = pd.DataFrame([{
                'TyreLife': tyre_age_A,
                'TyreLifePercent': tyre_age_A / 25.0, 
                'Avg_TrackTemp': 35.0,
                'Avg_AirTemp': 25.0,
                'TrackTemp_Delta': 0.0,
                'Peak_TyreEnergy': 3300.0,
                'Energy_Std': 10.0,
                'PaceTrend': 0.0,
                'Quad_Term': 0.0,
                'DegAcceleration': 0.0,
                'Compound': 'MEDIUM', # Or 'HARD'
                'CircuitArchetype': 'Mixed',
                'StintPhase': 'Early' if tyre_age_A <= 5 else ('Mid' if tyre_age_A <= 15 else 'Late')
            }])
            
            # 2. Get the predicted degradation from your XGBoost model
            predicted_delta = model.predict(race_state_A)[0]
            
            # 3. Base pace + ML prediction + Random chaos
            time_A = 92.5 + predicted_delta + random.uniform(-0.3, 0.3)
            
            # --- CAR B (RIVAL) USES SIMPLE MATH (For comparison) ---
            time_B = 93.0 + (tyre_age_B * 0.04) + random.uniform(-0.3, 0.3)
            
            # --- PIT STOPS ---
            if lap == pit_lap_A:
                time_A += random.uniform(20.5, 23.5) 
                tyre_age_A = 0
            else:
                tyre_age_A += 1
                
            if lap == pit_lap_B:
                time_B += random.uniform(20.5, 23.5) 
                tyre_age_B = 0
            else:
                tyre_age_B += 1
                
            total_time_A += time_A
            total_time_B += time_B
            
        if total_time_A < total_time_B:
            wins_A += 1
        else:
            wins_B += 1
            
    print("========================================")
    print(f"STRATEGY A (Pit Lap {pit_lap_A}) Win Rate: {(wins_A / num_simulations) * 100:.1f}%")
    print(f"STRATEGY B (Pit Lap {pit_lap_B}) Win Rate: {(wins_B / num_simulations) * 100:.1f}%")
    print("========================================")

# Run the Phase 5 Simulation!
run_phase5_simulation(total_laps=50, pit_lap_A=20, pit_lap_B=25, num_simulations=100)

STRATEGY A (Pit Lap 20) Win Rate: 100.0%
STRATEGY B (Pit Lap 25) Win Rate: 0.0%
